# IFCA Client

> The core abstraction for IFCA client.

In [ ]:
#| default_exp clients.ifca

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("ifca")
class ClientIFCA(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        

In [ ]:
#| export
@patch
def cluster_id_estimate(self: ClientIFCA):
    for model in self.k_models:
        model.to(self.device)
        model.eval()
    losses = []
    with torch.no_grad():
        for model in self.k_models:
            loss = 0.0
            for i, batch in enumerate(self.test_loader):
                batch = self.send_to_device(batch)
                X, y = batch[self.data_key], batch[self.label_key]
                y_pred = model(X)
                loss += self.criterion(y_pred, y).item()
            losses.append(loss)
    cluster_id = np.argmin(losses)
    return cluster_id

### Training

In [ ]:
#| export
@patch
def fit(self: ClientIFCA):
    self.cluster_id = self.cluster_id_estimate()
    self.model = copy.deepcopy(self.k_models[self.cluster_id].to(self.device))
    opt_fn, kwargs = self.optimizer
    optimizer = opt_fn(self.cfg)(self.model.parameters(), **kwargs)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            output = self.model(X)
            loss = self.criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

### Evaluation

In [ ]:
#| export
@patch
def evaluate_local(self: ClientIFCA, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []
    
    self.model = copy.deepcopy(self.k_models[self.cluster_id])
    self.model = self.model.to(self.device)
    self.model.eval()

    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    self.logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    self.logger.info(f"Average {loader} Metrics: {total_metrics}")
    return {"loss": avg_loss, "metrics": total_metrics}


### Saving

In [ ]:
#| hide
import torch.nn. functional as F
import torch
F.one_hot(torch.tensor(2), num_classes=4).tolist()

[0, 0, 1, 0]

In [ ]:
#| export
@patch
def save_state(self: ClientIFCA, save_to_disk= False):

    client_state = {
        'model': self.model.state_dict(),
        'optimizer': self.optimizer,
        'cluster_id': self.cluster_id,
        'one_hot_cluster_id': F.one_hot(torch.tensor(self.cluster_id), num_classes=self.cfg.algorithm.K).tolist(),
    }

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))
        torch.save(client_state, state_path)

    return client_state
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()